In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer, StringIndexer, OneHotEncoder, Imputer,
    VectorAssembler, StandardScaler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import unittest

In [19]:
# Start Spark session
spark = SparkSession.builder.appName("PowerliftingRegression").getOrCreate()

df = spark.read.csv("openpowerlifting.csv", header=True, inferSchema=True)

# Inspect column names and types
df.printSchema()
df.show(3)


root
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Event: string (nullable = true)
 |-- Equipment: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- AgeClass: string (nullable = true)
 |-- Division: string (nullable = true)
 |-- BodyweightKg: double (nullable = true)
 |-- WeightClassKg: string (nullable = true)
 |-- Squat1Kg: double (nullable = true)
 |-- Squat2Kg: double (nullable = true)
 |-- Squat3Kg: double (nullable = true)
 |-- Squat4Kg: double (nullable = true)
 |-- Best3SquatKg: double (nullable = true)
 |-- Bench1Kg: double (nullable = true)
 |-- Bench2Kg: double (nullable = true)
 |-- Bench3Kg: double (nullable = true)
 |-- Bench4Kg: double (nullable = true)
 |-- Best3BenchKg: double (nullable = true)
 |-- Deadlift1Kg: double (nullable = true)
 |-- Deadlift2Kg: double (nullable = true)
 |-- Deadlift3Kg: double (nullable = true)
 |-- Deadlift4Kg: double (nullable = true)
 |-- Best3DeadliftKg: double (nullable = true)
 |-- TotalKg: d

In [20]:
df = df.select(
    "Age", "Sex", "BodyweightKg", "Best3SquatKg", 
    "Best3BenchKg", "Best3DeadliftKg", "Equipment", 
    "Division", "Federation", "TotalKg"
).dropna(subset=["TotalKg"])
df

DataFrame[Age: double, Sex: string, BodyweightKg: double, Best3SquatKg: double, Best3BenchKg: double, Best3DeadliftKg: double, Equipment: string, Division: string, Federation: string, TotalKg: double]

# finding missing values

In [21]:
# Count nulls per column
null_counts = df.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()


+------+---+------------+------------+------------+---------------+---------+--------+----------+-------+
|   Age|Sex|BodyweightKg|Best3SquatKg|Best3BenchKg|Best3DeadliftKg|Equipment|Division|Federation|TotalKg|
+------+---+------------+------------+------------+---------------+---------+--------+----------+-------+
|601304|  0|        8777|      323015|       61362|         246749|        0|    7185|         0|      0|
+------+---+------------+------------+------------+---------------+---------+--------+----------+-------+



# Replacing missing values for numerical data

In [22]:
numeric_missing_cols = [
    "Age", "BodyweightKg", "Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg"
]
imputer = Imputer(
    inputCols=numeric_missing_cols,
    outputCols=[f"{col}_filled" for col in numeric_missing_cols]
)
df_imputed = imputer.fit(df).transform(df)

# Encoding categorical columns

In [24]:

categorical_cols = ["Sex", "Equipment", "Division", "Federation"]
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_enc")
    for col in categorical_cols
]

# Feature engineering with SQLTransformer

In [25]:
 sql = SQLTransformer(
    statement="""
        SELECT *, 
               Best3SquatKg_filled + Best3BenchKg_filled + Best3DeadliftKg_filled AS BestTotalEstimate
        FROM __THIS__
    """
)
df_sql = sql.transform(df_imputed)

df_sql.show(10)

+----+---+------------+------------+------------+---------------+---------+--------+----------+-------+----------+-------------------+-------------------+-------------------+----------------------+-----------------+
| Age|Sex|BodyweightKg|Best3SquatKg|Best3BenchKg|Best3DeadliftKg|Equipment|Division|Federation|TotalKg|Age_filled|BodyweightKg_filled|Best3SquatKg_filled|Best3BenchKg_filled|Best3DeadliftKg_filled|BestTotalEstimate|
+----+---+------------+------------+------------+---------------+---------+--------+----------+-------+----------+-------------------+-------------------+-------------------+----------------------+-----------------+
|29.0|  F|        59.8|       105.0|        55.0|          130.0|    Wraps|    F-OR|   GPC-AUS|  290.0|      29.0|               59.8|              105.0|               55.0|                 130.0|            290.0|
|29.0|  F|        58.5|       120.0|        67.5|          145.0|    Wraps|    F-OR|   GPC-AUS|  332.5|      29.0|               58.5|  

In [26]:
# Feature columns
features = [
    "Age_filled", "BodyweightKg_filled", "Best3SquatKg_filled",
    "Best3BenchKg_filled", "Best3DeadliftKg_filled", "BestTotalEstimate"
] + [f"{col}_enc" for col in categorical_cols]

assembler = VectorAssembler(inputCols=features, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Linear regression

In [27]:
lr = LinearRegression(featuresCol="scaledFeatures", labelCol="TotalKg")

In [28]:
# build pipeline and trainig
pipeline = Pipeline(stages=indexers + encoders + [imputer, sql, assembler, scaler, lr])
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)
predictions = model.transform(test_df)


In [29]:
# model evaluation
evaluator = RegressionEvaluator(labelCol="TotalKg", predictionCol="prediction")

rmse = evaluator.setMetricName("rmse").evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)
mae = evaluator.setMetricName("mae").evaluate(predictions)

In [30]:
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.2f}")

RMSE: 108.66
R²: 0.7076
MAE: 76.91


Модель работает достаточно хорошо с R² = 0.7076, что означает, что она объясняет около 70,76% дисперсии целевой переменной (TotalKg).

# Unittest

In [34]:
class TestPowerliftingPipeline(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.df = df
        cls.model = model
        cls.predictions = model.transform(df)
        cls.lr_model = cls.model.stages[-1]  # extract lr 

    def test_linear_regression_does_not_fail(self):
        try:
            self.model.transform(self.df)
        except Exception as e:
            self.fail(f"Linear Regression failed during transform: {e}")

    def test_prediction_column_exists(self):
        self.assertIn("prediction", self.predictions.columns)

    def test_output_shape_matches(self):
        self.assertEqual(self.df.count(), self.predictions.count())

    def test_model_has_coefficients(self):
        self.assertTrue(hasattr(self.lr_model, "coefficients"))
        self.assertGreater(len(self.lr_model.coefficients), 0)




In [35]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_linear_regression_does_not_fail (__main__.TestPowerliftingPipeline) ... ok
test_model_has_coefficients (__main__.TestPowerliftingPipeline) ... ok
test_output_shape_matches (__main__.TestPowerliftingPipeline) ... ok
test_prediction_column_exists (__main__.TestPowerliftingPipeline) ... ok

----------------------------------------------------------------------
Ran 4 tests in 2.830s

OK
